# 04 · Modelado y validación de demanda

## Objetivo

El notebook anterior dejó construido un panel semanal consistente para los mercados estratégicos del **Cluster 4**.

Ahora buscamos responder:

> **¿Cuánta demanda semanal esperamos por Tratamiento + Ciudad durante las próximas 1, 2, 3 y 4 semanas?**

La estrategia de modelado se diseña para soportar el objetivo posterior del proyecto: utilizar la demanda esperada como insumo de un optimizador de proveedores.

### Principios metodológicos

- Unidad de predicción: **Tratamiento + Ciudad + Semana**.
- Esquema **Direct Multi-Horizon**: H1, H2, H3 y H4.
- Cada horizonte se modela de forma independiente.
- Backtesting temporal, nunca aleatorio.
- Control explícito de **data leakage**.
- `ADI`, `CV²` y `TIPO_DEMANDA` se usan para diagnóstico, **no como features**.
- Comparación contra baselines simples y fuertes.
- Selección final por horizonte.

La versión final de la PoC trabaja operativamente con **CatBoost** y **Media Móvil de 8 semanas**.

## 0. Archivos requeridos

- `panel_cluster4_v2.csv`
- `metricas_demanda_cluster4_v2.csv`
- `calendario_venezuela_ml_2025_2027.xlsx`

En Google Colab, carga los archivos en `/content/` o ajusta las rutas.

In [ ]:
from pathlib import Path
import json
import importlib.util
import subprocess
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

if importlib.util.find_spec("catboost") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "catboost"
    ])

from catboost import CatBoostRegressor

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

def buscar_archivo(nombre):
    candidatos = [
        Path("/content") / nombre,
        Path("/mnt/data") / nombre,
        Path(nombre)
    ]
    return next((p for p in candidatos if p.exists()), None)

PANEL_PATH = buscar_archivo("panel_cluster4_v2.csv")
METRICAS_PATH = buscar_archivo("metricas_demanda_cluster4_v2.csv")
CALENDAR_PATH = buscar_archivo("calendario_venezuela_ml_2025_2027.xlsx")

archivos = {
    "Panel": PANEL_PATH,
    "Métricas demanda": METRICAS_PATH,
    "Calendario": CALENDAR_PATH
}

for nombre, ruta in archivos.items():
    print(f"{nombre:18}: {ruta}")

faltantes = [nombre for nombre, ruta in archivos.items() if ruta is None]

if faltantes:
    raise FileNotFoundError(
        "Faltan archivos requeridos: "
        + ", ".join(faltantes)
        + ". Ejecuta primero el notebook 03 o ajusta las rutas."
    )

## 1. Carga del panel definitivo

In [ ]:
df_modelo = pd.read_csv(
    PANEL_PATH,
    parse_dates=["FECHA_SEMANA"]
)

metricas_demanda = pd.read_csv(
    METRICAS_PATH
)

df_modelo = (
    df_modelo
    .sort_values(["SERIE_ID", "FECHA_SEMANA"])
    .reset_index(drop=True)
)

print("Filas panel:", f"{len(df_modelo):,}")
print("Series:", df_modelo["SERIE_ID"].nunique())
print(
    "Periodo:",
    df_modelo["FECHA_SEMANA"].min().date(),
    "→",
    df_modelo["FECHA_SEMANA"].max().date()
)
print(
    "% semanas con cero:",
    f"{df_modelo['N_SERVICIOS'].eq(0).mean():.2%}"
)

display(df_modelo.head())

## 2. Feature engineering histórico

En la fecha origen `t`, la semana `t` ya está cerrada y su demanda es conocida.

Por tanto:

- `LAG_0` = demanda observada en `t`.
- `LAG_1` = demanda en `t-1`.
- Las medias móviles incluyen la información disponible hasta `t`.

Esta semántica asegura que el modelo utiliza toda la información que existiría realmente al momento de pronosticar.

In [ ]:
fecha_min = df_modelo["FECHA_SEMANA"].min()

df_modelo["T"] = (
    (
        df_modelo["FECHA_SEMANA"] - fecha_min
    ).dt.days / 7
).astype(int)

df_modelo["LAG_0"] = df_modelo["N_SERVICIOS"]

for lag in [1, 2, 3, 4, 8]:
    df_modelo[f"LAG_{lag}"] = (
        df_modelo
        .groupby("SERIE_ID")["N_SERVICIOS"]
        .shift(lag)
    )

for ventana in [4, 8, 12]:
    df_modelo[f"MEDIA_MOVIL_{ventana}"] = (
        df_modelo
        .groupby("SERIE_ID")["N_SERVICIOS"]
        .transform(
            lambda x: x.rolling(
                ventana,
                min_periods=1
            ).mean()
        )
    )

for ventana in [4, 8]:
    df_modelo[f"STD_MOVIL_{ventana}"] = (
        df_modelo
        .groupby("SERIE_ID")["N_SERVICIOS"]
        .transform(
            lambda x: x.rolling(
                ventana,
                min_periods=2
            ).std()
        )
    )

for ventana in [4, 8]:
    df_modelo[f"SEMANAS_ACTIVAS_{ventana}"] = (
        df_modelo
        .groupby("SERIE_ID")["N_SERVICIOS"]
        .transform(
            lambda x: (x > 0)
            .rolling(
                ventana,
                min_periods=1
            )
            .sum()
        )
    )

In [ ]:
def calcular_semanas_desde_demanda(x):
    resultado = []
    semanas = None

    for valor in x:
        if valor > 0:
            semanas = 0
        elif semanas is not None:
            semanas += 1

        resultado.append(semanas)

    return pd.Series(
        resultado,
        index=x.index
    )

df_modelo["SEMANAS_DESDE_DEMANDA"] = (
    df_modelo
    .groupby("SERIE_ID")["N_SERVICIOS"]
    .transform(calcular_semanas_desde_demanda)
)

### Features históricas

- Lags 0, 1, 2, 3, 4 y 8.
- Medias móviles 4, 8 y 12.
- Desviaciones móviles 4 y 8.
- Semanas activas recientes.
- Semanas desde la última demanda.
- Tendencia temporal `T`.

No se utilizan descripciones textuales: las dimensiones categóricas ya están representadas por códigos e identificador de serie.

## 3. Targets directos H1–H4

In [ ]:
for h in [1, 2, 3, 4]:

    df_modelo[f"TARGET_H{h}"] = (
        df_modelo
        .groupby("SERIE_ID")["N_SERVICIOS"]
        .shift(-h)
    )

    df_modelo[f"FECHA_TARGET_H{h}"] = (
        df_modelo["FECHA_SEMANA"]
        + pd.to_timedelta(h, unit="W")
    )

### ¿Por qué Direct Multi-Horizon?

No se utiliza un enfoque recursivo en el que la predicción de H1 alimente H2 y así sucesivamente, porque el error se propagaría entre horizontes.

En su lugar:

- H1 aprende directamente `t → t+1`.
- H2 aprende directamente `t → t+2`.
- H3 aprende directamente `t → t+3`.
- H4 aprende directamente `t → t+4`.

Cada modelo puede además incorporar el calendario de la semana específica que intenta predecir.

## 4. Variables temporales de la semana objetivo

In [ ]:
for h in [1, 2, 3, 4]:

    fecha = df_modelo[f"FECHA_TARGET_H{h}"]

    df_modelo[f"MES_TARGET_H{h}"] = fecha.dt.month
    df_modelo[f"TRIMESTRE_TARGET_H{h}"] = fecha.dt.quarter

    semana = (
        fecha
        .dt.isocalendar()
        .week
        .astype(float)
    )

    df_modelo[f"SEMANA_ANIO_TARGET_H{h}"] = semana

    df_modelo[f"SEMANA_SIN_TARGET_H{h}"] = (
        np.sin(2 * np.pi * semana / 52)
    )

    df_modelo[f"SEMANA_COS_TARGET_H{h}"] = (
        np.cos(2 * np.pi * semana / 52)
    )

## 5. Calendario de la semana objetivo

In [ ]:
calendario = pd.read_excel(
    CALENDAR_PATH,
    sheet_name="Calendario_Semanal"
)

calendario["FECHA_SEMANA"] = pd.to_datetime(
    calendario["FECHA_SEMANA"]
)

variables_calendario = [
    "N_FESTIVOS_TOTAL",
    "N_FESTIVOS_LV",
    "DIAS_LABORABLES_LV",
    "N_DIAS_VACACIONALES_REF",
    "PCT_DIAS_VACACIONALES_REF",
    "TIENE_FESTIVO",
    "TIENE_VACACIONAL_REF",
    "ES_SEMANA_SANTA",
    "ES_CARNAVAL",
    "ES_NAVIDAD_FIN_ANIO"
]

for h in [1, 2, 3, 4]:

    cal = calendario[
        ["FECHA_SEMANA"] + variables_calendario
    ].copy()

    renombres = {
        "FECHA_SEMANA": f"FECHA_TARGET_H{h}"
    }

    for variable in variables_calendario:
        renombres[variable] = f"{variable}_H{h}"

    cal = cal.rename(columns=renombres)

    df_modelo = df_modelo.merge(
        cal,
        on=f"FECHA_TARGET_H{h}",
        how="left"
    )

columnas_calendario_target = [
    f"{v}_H{h}"
    for h in [1, 2, 3, 4]
    for v in variables_calendario
]

display(
    df_modelo[columnas_calendario_target]
    .isna()
    .sum()
    .to_frame("NULOS")
)

## 6. Dataset elegible para modelado

Las primeras semanas de cada serie no disponen de todos los lags requeridos. Estos valores nulos son estructurales y no se imputan arbitrariamente.

In [ ]:
features_historicas = [
    "LAG_0",
    "LAG_1",
    "LAG_2",
    "LAG_3",
    "LAG_4",
    "LAG_8",
    "MEDIA_MOVIL_4",
    "MEDIA_MOVIL_8",
    "MEDIA_MOVIL_12",
    "STD_MOVIL_4",
    "STD_MOVIL_8",
    "SEMANAS_ACTIVAS_4",
    "SEMANAS_ACTIVAS_8",
    "SEMANAS_DESDE_DEMANDA"
]

df_ml = (
    df_modelo
    .dropna(subset=features_historicas)
    .copy()
)

print("Filas ML:", f"{len(df_ml):,}")
print("Series ML:", df_ml["SERIE_ID"].nunique())
print(
    "Periodo:",
    df_ml["FECHA_SEMANA"].min().date(),
    "→",
    df_ml["FECHA_SEMANA"].max().date()
)

## 7. Features del modelo

### Categóricas
- `SERIE_ID`
- `CODIGO_TRATAMIENTO`
- `CODIGO_MUNICIPIO`

### Numéricas
- historia reciente;
- tendencia;
- calendario y temporalidad de la semana objetivo.

`ADI`, `CV²` y `TIPO_DEMANDA` quedan fuera de las features porque fueron calculados sobre la historia completa y se reservan para diagnóstico.

In [ ]:
features_base = [
    "SERIE_ID",
    "CODIGO_TRATAMIENTO",
    "CODIGO_MUNICIPIO",

    "LAG_0",
    "LAG_1",
    "LAG_2",
    "LAG_3",
    "LAG_4",
    "LAG_8",

    "MEDIA_MOVIL_4",
    "MEDIA_MOVIL_8",
    "MEDIA_MOVIL_12",

    "STD_MOVIL_4",
    "STD_MOVIL_8",

    "SEMANAS_ACTIVAS_4",
    "SEMANAS_ACTIVAS_8",
    "SEMANAS_DESDE_DEMANDA",

    "T"
]

categoricas = [
    "SERIE_ID",
    "CODIGO_TRATAMIENTO",
    "CODIGO_MUNICIPIO"
]

def features_horizonte(h):

    return features_base + [
        f"MES_TARGET_H{h}",
        f"TRIMESTRE_TARGET_H{h}",
        f"SEMANA_ANIO_TARGET_H{h}",
        f"SEMANA_SIN_TARGET_H{h}",
        f"SEMANA_COS_TARGET_H{h}",

        f"N_FESTIVOS_TOTAL_H{h}",
        f"N_FESTIVOS_LV_H{h}",
        f"DIAS_LABORABLES_LV_H{h}",
        f"N_DIAS_VACACIONALES_REF_H{h}",
        f"PCT_DIAS_VACACIONALES_REF_H{h}",
        f"TIENE_FESTIVO_H{h}",
        f"TIENE_VACACIONAL_REF_H{h}",
        f"ES_SEMANA_SANTA_H{h}",
        f"ES_CARNAVAL_H{h}",
        f"ES_NAVIDAD_FIN_ANIO_H{h}"
    ]

## 8. Rolling backtesting temporal

Se utilizan seis fechas de origen separadas por cuatro semanas:

- 2025-12-01
- 2025-12-29
- 2026-01-26
- 2026-02-23
- 2026-03-23
- 2026-04-20

### Regla anti-leakage

Para entrenar el horizonte `h` con origen `t`, solo se utilizan observaciones cuyo target ya habría sido conocido:

> `FECHA_TARGET_Hh <= t`

El test se construye en la fecha origen `t`.

In [ ]:
fechas_folds = [
    pd.Timestamp("2025-12-01"),
    pd.Timestamp("2025-12-29"),
    pd.Timestamp("2026-01-26"),
    pd.Timestamp("2026-02-23"),
    pd.Timestamp("2026-03-23"),
    pd.Timestamp("2026-04-20")
]

df_folds = pd.DataFrame({
    "FOLD": range(1, len(fechas_folds) + 1),
    "FECHA_ORIGEN": fechas_folds
})

for h in [1, 2, 3, 4]:
    df_folds[f"H{h}"] = (
        df_folds["FECHA_ORIGEN"]
        + pd.to_timedelta(h, unit="W")
    )

display(df_folds)

## 9. Cohorte común de evaluación

Para comparar horizontes sobre la misma población se toma la intersección de series evaluables en todos los folds y H1–H4.

El entrenamiento conserva toda la información histórica disponible; la restricción se aplica solo al conjunto de evaluación.

In [ ]:
conjuntos_series = []

for fecha_origen in fechas_folds:

    for h in [1, 2, 3, 4]:

        target = f"TARGET_H{h}"

        series_test = set(
            df_ml.loc[
                (df_ml["FECHA_SEMANA"] == fecha_origen) &
                (df_ml[target].notna()),
                "SERIE_ID"
            ]
        )

        conjuntos_series.append(series_test)

series_bt = set.intersection(*conjuntos_series)

print(
    "Series comunes para backtesting:",
    len(series_bt)
)

In [ ]:
resumen_folds = []

for num_fold, fecha_origen in enumerate(
    fechas_folds,
    start=1
):

    for h in [1, 2, 3, 4]:

        target = f"TARGET_H{h}"
        fecha_target = f"FECHA_TARGET_H{h}"

        train = df_ml[
            (df_ml[fecha_target] <= fecha_origen) &
            (df_ml[target].notna())
        ]

        test = df_ml[
            (df_ml["FECHA_SEMANA"] == fecha_origen) &
            (df_ml["SERIE_ID"].isin(series_bt)) &
            (df_ml[target].notna())
        ]

        resumen_folds.append({
            "FOLD": num_fold,
            "ORIGEN": fecha_origen,
            "HORIZONTE": f"H{h}",
            "TRAIN_FILAS": len(train),
            "TRAIN_SERIES": train["SERIE_ID"].nunique(),
            "TEST_FILAS": len(test),
            "TEST_SERIES": test["SERIE_ID"].nunique()
        })

resumen_folds = pd.DataFrame(resumen_folds)

display(resumen_folds)

### Resultado esperado

Con el panel V2 se obtuvieron **50 series comunes**, frente a 48 en la primera versión. Esto confirmó que la corrección de ceros terminales mejoró la calidad del backtesting.

## 10. Métricas de evaluación

- **MAE:** error absoluto medio en número de servicios.
- **RMSE:** penaliza errores grandes.
- **WAPE:** error absoluto agregado relativo al volumen real.
- **Bias:** identifica sobreestimación o subestimación sistemática.

No se utiliza MAPE como métrica principal porque existen muchas observaciones reales iguales a cero.

In [ ]:
def calcular_metricas(grupo):

    real = grupo["REAL"].to_numpy(dtype=float)
    pred = grupo["PREDICCION"].to_numpy(dtype=float)

    mae = np.mean(np.abs(real - pred))

    rmse = np.sqrt(
        np.mean((real - pred) ** 2)
    )

    denominador = np.sum(np.abs(real))

    wape = (
        np.sum(np.abs(real - pred))
        / denominador * 100
        if denominador > 0
        else np.nan
    )

    bias = np.mean(pred - real)

    return pd.Series({
        "N": len(grupo),
        "MAE": mae,
        "RMSE": rmse,
        "WAPE": wape,
        "BIAS": bias
    })

## 11. Baselines

Se comparan tres referencias simples:

- `NAIVE`: repetir la última semana conocida.
- `MEDIA_4`: media de las últimas 4 semanas.
- `MEDIA_8`: media de las últimas 8 semanas.

Un modelo ML solo se justifica si mejora una referencia razonable.

In [ ]:
detalle_baseline = []

for num_fold, fecha_origen in enumerate(
    fechas_folds,
    start=1
):

    for h in [1, 2, 3, 4]:

        target = f"TARGET_H{h}"

        test = df_ml[
            (df_ml["FECHA_SEMANA"] == fecha_origen) &
            (df_ml["SERIE_ID"].isin(series_bt)) &
            (df_ml[target].notna())
        ].copy()

        predicciones = {
            "NAIVE": "LAG_0",
            "MEDIA_4": "MEDIA_MOVIL_4",
            "MEDIA_8": "MEDIA_MOVIL_8"
        }

        for modelo, columna in predicciones.items():

            tmp = test[
                [
                    "SERIE_ID",
                    "CODIGO_TRATAMIENTO",
                    "CODIGO_MUNICIPIO",
                    target,
                    columna
                ]
            ].copy()

            tmp["FOLD"] = num_fold
            tmp["FECHA_ORIGEN"] = fecha_origen
            tmp["HORIZONTE"] = f"H{h}"
            tmp["MODELO"] = modelo

            tmp = tmp.rename(
                columns={
                    target: "REAL",
                    columna: "PREDICCION"
                }
            )

            detalle_baseline.append(tmp)

detalle_baseline = pd.concat(
    detalle_baseline,
    ignore_index=True
)

metricas_baseline = (
    detalle_baseline
    .groupby(["MODELO", "HORIZONTE"])
    .apply(
        calcular_metricas,
        include_groups=False
    )
    .reset_index()
)

display(
    metricas_baseline
    .sort_values(["HORIZONTE", "WAPE"])
)

### Hallazgo

`MEDIA_8` es el baseline más fuerte en los cuatro horizontes. Esto indica que la demanda contiene ruido semanal suficiente para que un suavizado de ocho semanas sea más robusto que repetir la última observación.

## 12. Configuraciones CatBoost evaluadas

In [ ]:
configuraciones_catboost = {

    "CB_ACTUAL": {
        "iterations": 500,
        "depth": 6,
        "learning_rate": 0.05,
        "loss_function": "RMSE",
        "l2_leaf_reg": 3
    },

    "CB_SHALLOW": {
        "iterations": 600,
        "depth": 4,
        "learning_rate": 0.05,
        "loss_function": "RMSE",
        "l2_leaf_reg": 5
    },

    "CB_MAE": {
        "iterations": 600,
        "depth": 5,
        "learning_rate": 0.05,
        "loss_function": "MAE",
        "l2_leaf_reg": 5
    },

    "CB_POISSON": {
        "iterations": 600,
        "depth": 5,
        "learning_rate": 0.05,
        "loss_function": "Poisson",
        "l2_leaf_reg": 5
    }
}

display(pd.DataFrame(configuraciones_catboost).T)

### Por qué se probaron estas variantes

- `CB_ACTUAL`: configuración inicial.
- `CB_SHALLOW`: menor profundidad y mayor regularización.
- `CB_MAE`: objetivo alineado con error absoluto y WAPE.
- `CB_POISSON`: alternativa natural para una variable de conteo.

No se hace una búsqueda masiva de hiperparámetros para evitar sobreoptimización sobre solo seis folds.

## 13. Rolling backtest CatBoost

In [ ]:
resultados_catboost = []

for nombre_modelo, params in configuraciones_catboost.items():

    print("Ejecutando:", nombre_modelo)

    for num_fold, fecha_origen in enumerate(
        fechas_folds,
        start=1
    ):

        for h in [1, 2, 3, 4]:

            target = f"TARGET_H{h}"
            fecha_target = f"FECHA_TARGET_H{h}"
            features = features_horizonte(h)

            train = df_ml[
                (df_ml[fecha_target] <= fecha_origen) &
                (df_ml[target].notna())
            ].copy()

            test = df_ml[
                (df_ml["FECHA_SEMANA"] == fecha_origen) &
                (df_ml["SERIE_ID"].isin(series_bt)) &
                (df_ml[target].notna())
            ].copy()

            X_train = train[features].copy()
            y_train = train[target].copy()

            X_test = test[features].copy()
            y_test = test[target].copy()

            for col in categoricas:
                X_train[col] = X_train[col].astype(str)
                X_test[col] = X_test[col].astype(str)

            modelo = CatBoostRegressor(
                **params,
                random_seed=42,
                verbose=False
            )

            modelo.fit(
                X_train,
                y_train,
                cat_features=categoricas
            )

            pred = np.maximum(
                modelo.predict(X_test),
                0
            )

            tmp = test[
                [
                    "SERIE_ID",
                    "CODIGO_TRATAMIENTO",
                    "CODIGO_MUNICIPIO"
                ]
            ].copy()

            tmp["FOLD"] = num_fold
            tmp["FECHA_ORIGEN"] = fecha_origen
            tmp["HORIZONTE"] = f"H{h}"
            tmp["MODELO"] = nombre_modelo
            tmp["REAL"] = y_test.to_numpy()
            tmp["PREDICCION"] = pred

            resultados_catboost.append(tmp)

detalle_catboost = pd.concat(
    resultados_catboost,
    ignore_index=True
)

metricas_catboost = (
    detalle_catboost
    .groupby(["MODELO", "HORIZONTE"])
    .apply(
        calcular_metricas,
        include_groups=False
    )
    .reset_index()
)

comparacion_modelos = pd.concat(
    [metricas_baseline, metricas_catboost],
    ignore_index=True
)

display(
    comparacion_modelos
    .sort_values(["HORIZONTE", "WAPE"])
)

## 14. Comparación visual de WAPE

In [ ]:
modelos_grafico = [
    "MEDIA_8",
    "CB_ACTUAL",
    "CB_SHALLOW",
    "CB_MAE",
    "CB_POISSON"
]

tmp = comparacion_modelos[
    comparacion_modelos["MODELO"].isin(modelos_grafico)
].copy()

pivot_wape = (
    tmp
    .pivot(
        index="HORIZONTE",
        columns="MODELO",
        values="WAPE"
    )
    .reindex(["H1", "H2", "H3", "H4"])
)

ax = pivot_wape.plot(
    kind="bar",
    figsize=(11, 5.5)
)

ax.set_title("WAPE por horizonte y modelo")
ax.set_xlabel("Horizonte")
ax.set_ylabel("WAPE (%)")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=0)
plt.legend(title="Modelo")

plt.show()

### Hallazgo

`CB_MAE` fue la configuración CatBoost más consistente y obtuvo el menor WAPE entre las variantes CatBoost en los cuatro horizontes durante la fase de tuning.

Además, permite mantener una misma configuración para H1–H3.

## 15. Backtest definitivo: CB_MAE vs MEDIA_8

La corrección del panel temporal modificó ligeramente la comparación, especialmente en H4.

La decisión final se toma únicamente entre:

- `CB_MAE`
- `MEDIA_8`

sobre las **50 series comunes**.

In [ ]:
params_cb_mae = configuraciones_catboost["CB_MAE"]

detalle_final = pd.concat(
    [
        detalle_baseline[
            detalle_baseline["MODELO"] == "MEDIA_8"
        ],
        detalle_catboost[
            detalle_catboost["MODELO"] == "CB_MAE"
        ]
    ],
    ignore_index=True
)

metricas_finales = (
    detalle_final
    .groupby(["MODELO", "HORIZONTE"])
    .apply(
        calcular_metricas,
        include_groups=False
    )
    .reset_index()
)

display(
    metricas_finales
    .sort_values(["HORIZONTE", "WAPE"])
)

### Resultados de referencia de la PoC

| Horizonte | Modelo | MAE | RMSE | WAPE | Bias |
|---|---|---:|---:|---:|---:|
| H1 | **CB_MAE** | **1,054** | **1,999** | **62,50 %** | -0,009 |
| H1 | MEDIA_8 | 1,224 | 2,032 | 72,55 % | 0,131 |
| H2 | **CB_MAE** | **1,175** | **1,781** | **58,96 %** | -0,247 |
| H2 | MEDIA_8 | 1,235 | 1,834 | 61,98 % | -0,175 |
| H3 | **CB_MAE** | **1,215** | **1,753** | **69,71 %** | -0,066 |
| H3 | MEDIA_8 | 1,243 | 1,784 | 71,30 % | 0,075 |
| H4 | CB_MAE | 1,286 | 1,979 | 70,04 % | -0,046 |
| H4 | **MEDIA_8** | **1,279** | **1,894** | **69,62 %** | -0,019 |

La mejora de CatBoost es clara en H1, moderada en H2, débil en H3 y desaparece en H4.

## 16. Estabilidad por fold

In [ ]:
metricas_fold = (
    detalle_final
    .groupby(
        ["FOLD", "HORIZONTE", "MODELO"]
    )
    .apply(
        calcular_metricas,
        include_groups=False
    )
    .reset_index()
)

ganadores_fold = (
    metricas_fold
    .pivot_table(
        index=["FOLD", "HORIZONTE"],
        columns="MODELO",
        values="WAPE"
    )
    .reset_index()
)

ganadores_fold["DIF_WAPE"] = (
    ganadores_fold["MEDIA_8"]
    - ganadores_fold["CB_MAE"]
)

ganadores_fold["GANADOR"] = np.where(
    ganadores_fold["CB_MAE"]
    < ganadores_fold["MEDIA_8"],
    "CB_MAE",
    "MEDIA_8"
)

resumen_ganadores = (
    ganadores_fold
    .groupby(["HORIZONTE", "GANADOR"])
    .size()
    .reset_index(name="FOLDS")
)

display(resumen_ganadores)

In [ ]:
pivot_folds = (
    ganadores_fold
    .pivot(
        index="FOLD",
        columns="HORIZONTE",
        values="DIF_WAPE"
    )
    .reindex(columns=["H1", "H2", "H3", "H4"])
)

ax = pivot_folds.plot(
    kind="bar",
    figsize=(11, 5.5)
)

ax.axhline(0, linewidth=1)

ax.set_title(
    "Ventaja de CatBoost sobre Media 8 por fold"
)
ax.set_xlabel("Fold")
ax.set_ylabel(
    "WAPE Media 8 - WAPE CatBoost (pp)"
)
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=0)
plt.legend(title="Horizonte")

plt.show()

### Lectura de estabilidad

- H1: CatBoost gana **4 de 6 folds**.
- H2: empate **3 vs 3**.
- H3: empate **3 vs 3**.
- H4: Media 8 gana **4 de 6 folds**.

La decisión no depende solo del número de folds ganados; se considera también la magnitud del error agregado.

## 17. Diagnóstico por tipo de demanda

In [ ]:
tipo_serie = metricas_demanda[
    ["SERIE_ID", "TIPO_DEMANDA"]
].drop_duplicates()

detalle_tipo = detalle_final.merge(
    tipo_serie,
    on="SERIE_ID",
    how="left"
)

metricas_tipo = (
    detalle_tipo
    .groupby(
        ["MODELO", "HORIZONTE", "TIPO_DEMANDA"]
    )
    .apply(
        calcular_metricas,
        include_groups=False
    )
    .reset_index()
)

display(
    metricas_tipo
    .sort_values(
        ["TIPO_DEMANDA", "HORIZONTE", "WAPE"]
    )
)

### Hallazgo

CatBoost aporta una mejora más clara en las series **Smooth**, que concentran la mayor parte del volumen.

En las series **Intermittent**, la diferencia frente a Media 8 es menos consistente.

Para mantener una arquitectura simple y evitar sobreajuste con pocas observaciones por serie, no se implementa un selector de algoritmo por mercado. La decisión se mantiene a nivel de horizonte.

## 18. Selección final por horizonte

In [ ]:
seleccion_modelo = {
    "H1": {
        "modelo": "CB_MAE",
        "tipo": "catboost",
        "parametros": params_cb_mae
    },
    "H2": {
        "modelo": "CB_MAE",
        "tipo": "catboost",
        "parametros": params_cb_mae
    },
    "H3": {
        "modelo": "CB_MAE",
        "tipo": "catboost",
        "parametros": params_cb_mae
    },
    "H4": {
        "modelo": "MEDIA_8",
        "tipo": "baseline",
        "ventana": 8
    }
}

display(pd.DataFrame(seleccion_modelo).T)

### Modelo seleccionado

> **H1 → CatBoost MAE**  
> **H2 → CatBoost MAE**  
> **H3 → CatBoost MAE**  
> **H4 → Media Móvil de 8 semanas**

H4 usa Media 8 porque presenta ligeramente menor WAPE, MAE y RMSE, y gana 4 de los 6 folds.

En H3 la mejora de CatBoost es pequeña, pero se mantiene porque conserva mejores métricas agregadas y no añade complejidad tecnológica adicional.

## 19. Incertidumbre del forecast

El optimizador de proveedores no debe interpretar el forecast como una cantidad cierta.

Para planificación de capacidad se construyen escenarios a partir de los errores de subestimación observados en el rolling backtest:

- **BASE:** demanda esperada.
- **P80:** escenario de capacidad razonablemente conservador.
- **P90:** escenario de estrés.

Los buffers se normalizan por una escala basada en `MEDIA_MOVIL_8`.

In [ ]:
detalle_seleccionado = detalle_final[
    detalle_final.apply(
        lambda x:
        x["MODELO"]
        == (
            "CB_MAE"
            if x["HORIZONTE"] in ["H1", "H2", "H3"]
            else "MEDIA_8"
        ),
        axis=1
    )
].copy()

escala_bt = df_ml[
    [
        "SERIE_ID",
        "FECHA_SEMANA",
        "MEDIA_MOVIL_8"
    ]
].rename(
    columns={
        "FECHA_SEMANA": "FECHA_ORIGEN"
    }
)

detalle_seleccionado = detalle_seleccionado.merge(
    escala_bt,
    on=["SERIE_ID", "FECHA_ORIGEN"],
    how="left"
)

detalle_seleccionado["ESCALA"] = np.maximum(
    detalle_seleccionado["MEDIA_MOVIL_8"],
    1
)

detalle_seleccionado["ERROR_SUPERIOR"] = (
    detalle_seleccionado["REAL"]
    - detalle_seleccionado["PREDICCION"]
)

detalle_seleccionado["ERROR_SUPERIOR_NORMALIZADO"] = (
    detalle_seleccionado["ERROR_SUPERIOR"]
    / detalle_seleccionado["ESCALA"]
)

buffers_horizonte = (
    detalle_seleccionado
    .groupby("HORIZONTE")[
        "ERROR_SUPERIOR_NORMALIZADO"
    ]
    .quantile([0.80, 0.90])
    .unstack()
    .reset_index()
    .rename(
        columns={
            0.80: "BUFFER_P80",
            0.90: "BUFFER_P90"
        }
    )
)

display(buffers_horizonte)

### Buffers obtenidos en el desarrollo

| Horizonte | Buffer P80 | Buffer P90 |
|---|---:|---:|
| H1 | 0,515 | 0,947 |
| H2 | 0,824 | 1,342 |
| H3 | 0,810 | 1,086 |
| H4 | 0,756 | 1,462 |

Estos valores **no son porcentajes para multiplicar directamente por la predicción**.

La construcción correcta es:

`DEMANDA_ESCENARIO = PREDICCION_BASE + BUFFER × ESCALA`

con:

`ESCALA = max(MEDIA_MOVIL_8, 1)`

## 20. Exportar configuración validada

In [ ]:
OUTPUT_CONFIG = Path(
    "/mnt/data/config_modelo_demanda.json"
)

OUTPUT_BUFFERS = Path(
    "/mnt/data/buffers_demanda.csv"
)

config_exportable = {
    "granularidad": "TRATAMIENTO_CIUDAD_SEMANA",
    "frecuencia": "W-MON",
    "horizontes": seleccion_modelo,
    "features_historicas": features_historicas,
    "features_categoricas": categoricas,
    "n_series_backtest": len(series_bt),
    "fechas_folds": [
        str(f.date()) for f in fechas_folds
    ]
}

with open(
    OUTPUT_CONFIG,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        config_exportable,
        f,
        ensure_ascii=False,
        indent=2
    )

buffers_horizonte.to_csv(
    OUTPUT_BUFFERS,
    index=False
)

print(f"Configuración: {OUTPUT_CONFIG}")
print(f"Buffers:       {OUTPUT_BUFFERS}")

# Conclusiones

El modelado deja cerrada la primera pieza predictiva de la solución:

1. Forecast semanal global por **Tratamiento + Ciudad**.
2. Enfoque **Direct Multi-Horizon H1–H4**.
3. Rolling backtesting con control de leakage.
4. Cohorte común de 50 series en la validación final.
5. `MEDIA_8` como baseline principal.
6. `CB_MAE` como mejor configuración CatBoost.
7. Selección final:
   - H1 → CatBoost MAE
   - H2 → CatBoost MAE
   - H3 → CatBoost MAE
   - H4 → Media 8
8. Buffers P80/P90 para trasladar incertidumbre a la futura planificación de capacidad.
9. El modelo **no decide proveedores**: pronostica demanda. La decisión de proveedor corresponde al módulo posterior de optimización.

El siguiente notebook deja de experimentar y se concentra en:

- entrenar con todo el histórico disponible;
- serializar H1–H3;
- parametrizar H4;
- guardar metadatos;
- validar carga e inferencia independiente.

➡️ **Siguiente notebook: `05_Entrenamiento_Exportacion.ipynb`**